# Population gradients of the log loss at designed parameter configurations (run 60 architecture)

No training here. The notebook builds parameter configurations of the run 60 architecture (62-60 ReLU net on the depth-5 COB-ODT, no
biases, direct readout) and computes the population gradient of the logistic loss at each of them (full-batch over 400k points), to read
what gradient descent would do first from such a point. Two configurations for now:

1. **Ideal first layer, random second layer.** W1 = the 62 one-sided features +-u_j with norms base x 2^(-level); W2 Gaussian; W_out
   alternating +-1. The design of W2 in runs 37-62 made the *probes* align to the ODT normals; the question here is the converse: given the
   ideal features, does the gradient on W2 build the designed second layer (vetoes on the wrong side of a node, readout-aligned leaf
   weights)?
2. **Run 60's W2 and W_out, first layer collapsed onto v.** Every first-layer row is alpha v with v = u_15 + ... + u_30 and alpha chosen to
   minimise the training log loss; the 32 cue rows get a small extra component epsilon x pole x u_m. How does the gradient on the probes
   look then: does each probe get pushed toward its designed node, and how does that push compare with the pull toward the leaves?

Conventions: a "push" is minus the gradient (the direction of a descent step) projected on a unit vector; `side_j(m)` is the side of
node j on which leaf m lies; a unit's *cue reading* of leaf m is `c_u(m) = sum over the two cue rows of m of W2[u, row] x pole x ||row||`;
its *side-contrast* on an internal node j is `C_u(j) = sum over leaves m under j of side_j(m) c_u(m)`, and its *half-reading* on the
side `pole` of j is `S_u(j, pole) = sum over leaves m under j with side_j(m) = pole of c_u(m)` (so `C = S(+) - S(-)`).


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle, torch
%load_ext autoreload
%autoreload 2
from layer_lenses.designed_init import NODE_LEVEL, node_ancestors, HIERARCHY_TARGETS, cue_row_identity
from layer_lenses.init_sweep import SweepRunSpec, build_data, load_tree
np.set_printoptions(linewidth=200, precision=3, suppress=True); pd.set_option("display.width", 250); dev = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS = Path("../results/designed_init"); RUN60 = RESULTS / "hd2_run60_sgd_lr1_1000"; tree = load_tree(RUN60 / "tree.npz"); U = tree.w_list
z60 = np.load(RUN60 / "replay/weights.npz"); meta60 = pickle.load(open(RUN60 / "replay/meta.pkl", "rb"))
W1_60, W2_60, Wo_60 = (z60[k][0].astype(np.float64) for k in ["w__hidden_layers.0.weight", "w__hidden_layers.1.weight", "w__output_layer.weight"])       # run 60's init
CUE_ROWS = list(meta60["design"]["cue_rows"]); PROBE_ROWS = list(range(30)); unode = np.array(meta60["design"]["unit_node"]); ups = np.array(meta60["design"]["unit_path_sign"])
CUE_ID = cue_row_identity(W1_60, tree, CUE_ROWS); ROW_ID = {r: (n, p) for (n, p), r in CUE_ID.items()}; ROW_ID.update({p: HIERARCHY_TARGETS[p] for p in PROBE_ROWS})     # row -> (node, pole) by design
N_POINTS = 400_000
x, y = build_data(SweepRunSpec(tree_seed=5178, data_seed=31337, init_seed=0, batch_size=1, n_train=N_POINTS), tree)
X = torch.from_numpy(x).float().to(dev); Y = torch.from_numpy(y.astype(np.float32)).to(dev); A = x @ U.T
v = U[15:31].sum(0)                                                                                          # the sum of the level-4 normals (|v| = 4)
def side_of(j, m):
    """Side (+1 / -1) of internal node j on which node m (a descendant) lies."""
    c = m
    while (c - 1) // 2 != j: c = (c - 1) // 2
    return +1 if c == 2 * j + 2 else -1
LEAVES_UNDER = {j: [m for m in range(15, 31) if j in node_ancestors(m)] for j in range(15)}

class _ReLUSub1(torch.autograd.Function):
    """ReLU whose subgradient at 0 is 1 (torch.relu uses 0), so a layer sitting exactly at 0 still passes gradient."""
    @staticmethod
    def forward(ctx, z): ctx.save_for_backward(z); return z.clamp(min=0)
    @staticmethod
    def backward(ctx, g): (z,) = ctx.saved_tensors; return g * (z >= 0).to(g.dtype)
def population_gradient(W1, W2, Wo, chunk=100_000, relu_sub1=False):
    """Mean logistic loss over the sample and its gradients wrt W1, W2, Wout (the population gradient, up to sampling). relu_sub1: ReLU'(0) = 1."""
    act = _ReLUSub1.apply if relu_sub1 else torch.relu
    P = [torch.as_tensor(np.asarray(w, dtype=np.float32), device=dev).requires_grad_(True) for w in (W1, W2, Wo)]; total = 0.0
    for i in range(0, len(X), chunk):
        xb, yb = X[i:i + chunk], Y[i:i + chunk]; f = (act(act(xb @ P[0].T) @ P[1].T) @ P[2].T).squeeze(1)
        l = torch.nn.functional.softplus(-yb * f).sum() / len(X); l.backward(); total += float(l)
    return total, [p.grad.cpu().numpy().astype(np.float64) for p in P]

def cue_readings(W1, W2):
    """c[u, m]: unit u's pole-signed, norm-weighted reading of leaf m through the cue rows (by their designed identity)."""
    nrm = np.linalg.norm(W1, axis=1); c = np.zeros((W2.shape[0], 31))
    for (m, pole), r in CUE_ID.items(): c[:, m] += W2[:, r] * pole * nrm[r]
    return c
def half_readings(c):
    """S[u, j, k] for internal j and k in (0: pole +, 1: pole -): the unit's reading summed over the leaves under j on that side; contrast C = S[..., 0] - S[..., 1]."""
    S = np.zeros((c.shape[0], 15, 2))
    for j in range(15):
        for m in LEAVES_UNDER[j]: S[:, j, 0 if side_of(j, m) > 0 else 1] += c[:, m]
    return S
def ideal_first_layer(base=np.sqrt(2.0)):
    """62 rows at +-u_j in run 60's row order (probes 0-29 on their designed (node, pole), cues 30-61 on theirs), norm base x 2^(-level)."""
    W1 = np.zeros((62, 100))
    for r, (n, p) in ROW_ID.items(): W1[r] = p * U[n] * base * 2.0 ** (-NODE_LEVEL[n])
    return W1
feature_label = np.array([f"{'+' if ROW_ID[r][1] > 0 else '-'}u{ROW_ID[r][0]}" for r in range(62)]); feature_level = np.array([NODE_LEVEL[ROW_ID[r][0]] for r in range(62)])
print(f"device {dev}; {N_POINTS} points; run 60 init loaded (W2 {W2_60.shape}, |W2 row| {np.linalg.norm(W2_60, axis=1).mean():.3f}, Wout entries {np.unique(np.round(Wo_60, 3))}); feature labels by design: {feature_label[:4]} ... {feature_label[-2:]}")


## 1. Ideal first layer, random W2, alternating readout: does the gradient on W2 build the designed second layer?

W1 = `ideal_first_layer()` (norm sqrt 2 x 2^(-level), so 1.41 at the root, 0.088 for the cues), W2 ~ N(0, 2/62) (He; |row| about sqrt 2, as
the design), W_out[u] = (-1)^u. Lens prediction for the descent direction on W2 at such a point (f = 0, resid = -y/2):
`-dL/dW2[u, (j, pole)] ~ Wout[u] x S_u(j, pole) x E[relu(pole a_j) on that side]` for an internal feature, and `~ Wout[u] x pole x E|a_m|`
for a leaf feature. So every unit wants leaf features with the sign of its readout (a label-aligned leaf reading), and it wants the feature of
side `pole` of node j with the sign of its own half-reading on that side times its readout: a **veto** (negative weight) on the side where the
unit's leaf reading is opposed to its readout, a positive weight on the other side. The cells test this on the measured gradient, and on its **label part**: the gradient at f -> 0 with the same gates (the residual -y sigma(-y f)
replaced by -y/2), which is what the lens formula models. At a random W2 with a +-1 readout the logit is not small, and the residual then
carries a label-independent part that pushes all of a unit's input weights the same way; the label part removes it.


In [ ]:
def analyse_random_w2(seed, base=np.sqrt(2.0), label_part=False):
    rng = np.random.default_rng(seed); W1 = ideal_first_layer(base); W2 = rng.standard_normal((60, 62)) * np.sqrt(2 / 62); Wo = np.array([[(-1.0) ** u for u in range(60)]])
    loss, (G1, G2, Go) = population_gradient(W1, W2, Wo)
    _, (_, G2s, _) = population_gradient(W1, W2, 1e-3 * Wo); D2 = -G2 if not label_part else -G2s * 1e3                      # descent on W2: actual, or its label part (the gradient at f -> 0, same gates; exact since dL/dW2 is linear in Wout at fixed gates)
    c = cue_readings(W1, W2); S = half_readings(c); C = S[:, :, 0] - S[:, :, 1]
    # leaf columns: descent sign vs Wout[u] x pole
    leaf_cols = [r for r in range(62) if feature_level[r] == 4]; pred_leaf = np.array([[Wo[0, u] * ROW_ID[r][1] for r in leaf_cols] for u in range(60)])
    leaf_agree = float((np.sign(D2[:, leaf_cols]) == np.sign(pred_leaf)).mean())
    # internal columns: descent vs Wout[u] x S_u(j, pole); veto statistic: opposite signs of the descent on the two poles of j, with the negative one on the side where Wout x S < 0
    int_cols = [r for r in range(62) if feature_level[r] < 4]; pred_int = np.array([[Wo[0, u] * S[u, ROW_ID[r][0], 0 if ROW_ID[r][1] > 0 else 1] for r in int_cols] for u in range(60)])
    corr_by_level = {l: np.corrcoef(D2[:, [r for r in int_cols if feature_level[r] == l]].ravel(), pred_int[:, [i for i, r in enumerate(int_cols) if feature_level[r] == l]].ravel())[0, 1] for l in range(4)}
    veto = {l: [] for l in range(4)}; veto_pred = {l: [] for l in range(4)}
    for u in range(60):
        for j in range(15):
            rp = [r for r in range(62) if ROW_ID[r] == (j, 1)][0]; rm = [r for r in range(62) if ROW_ID[r] == (j, -1)][0]; dp, dm = D2[u, rp], D2[u, rm]
            veto[NODE_LEVEL[j]].append(dp * dm < 0)                                                                                       # one side pushed negative, the other positive
            veto_pred[NODE_LEVEL[j]].append(dp * dm < 0 and np.sign(dp) == np.sign(Wo[0, u] * S[u, j, 0]) and np.sign(dm) == np.sign(Wo[0, u] * S[u, j, 1]))
    rms_by_level = {l: np.sqrt(np.mean(D2[:, [r for r in range(62) if feature_level[r] == l]] ** 2)) for l in range(5)}
    f_rms = float(np.sqrt(np.mean(forward_logit(W1, W2, Wo) ** 2)))
    rec = {"seed": seed, "gradient": "label part (f -> 0)" if label_part else "actual", "loss": round(loss, 4), "logit rms": round(f_rms, 2), "|G1| / |G2| / |Go|": f"{np.linalg.norm(G1):.2e} / {np.linalg.norm(G2):.2e} / {np.linalg.norm(Go):.2e}", "leaf columns: sign = Wout x pole": round(leaf_agree, 3),
           "internal columns: corr(descent, Wout x S) by level 0/1/2/3": " / ".join(f"{corr_by_level[l]:.2f}" for l in range(4)), "pairs (u, j) with opposite-sign descent on the two poles, by level": " / ".join(f"{np.mean(veto[l]):.2f}" for l in range(4)),
           "... and with the predicted sides": " / ".join(f"{np.mean(veto_pred[l]):.2f}" for l in range(4)), "descent rms by feature level 0/1/2/3/4": " / ".join(f"{rms_by_level[l]:.1e}" for l in range(5))}
    return rec, (W1, W2, Wo, D2, S, C, G1)
def forward_logit(W1, W2, Wo, n=100_000):
    with torch.no_grad(): return (torch.relu(torch.relu(X[:n] @ torch.as_tensor(W1, dtype=torch.float32, device=dev).T) @ torch.as_tensor(W2, dtype=torch.float32, device=dev).T) @ torch.as_tensor(Wo, dtype=torch.float32, device=dev).T).squeeze(1).cpu().numpy()
recs = []; keep = {}
for seed in range(5):
    rec, _ = analyse_random_w2(seed); recs.append(rec); rec, keep[seed] = analyse_random_w2(seed, label_part=True); recs.append(rec)
print(pd.DataFrame(recs).to_string(index=False)); print("\nchance level for 'opposite-sign descent on the two poles' is 0.50")


In [ ]:
# The descent direction on W2 for seed 0, in the feature basis (units x features sorted by node), and the lens prediction next to it.
W1, W2, Wo, D2, S, C, G1 = keep[0]; order = np.lexsort((-np.array([ROW_ID[r][1] for r in range(62)]), np.array([ROW_ID[r][0] for r in range(62)])))
pred = np.zeros_like(D2)
for u in range(60):
    for r in range(62):
        n, p = ROW_ID[r]; pred[u, r] = Wo[0, u] * (p if n >= 15 else S[u, n, 0 if p > 0 else 1])
fig, ax = plt.subplots(1, 2, figsize=(18, 7))
for k, (M, title) in enumerate([(D2, "descent direction -dL/dW2, label part (seed 0)"), (pred, "lens prediction: Wout x (pole for leaves, S_u(j, pole) for internal nodes)")]):
    Ms = M[:, order] / np.abs(M[:, order]).max(0, keepdims=True).clip(1e-12); im = ax[k].imshow(Ms, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"); ax[k].set_title(title + " (each column scaled to max 1)", fontsize=9)
    ax[k].set_xticks(range(62)); ax[k].set_xticklabels(feature_label[order], rotation=90, fontsize=6); ax[k].set_ylabel("unit")
    for kk in np.cumsum([(feature_level[order] == l).sum() for l in range(4)]): ax[k].axvline(kk - 0.5, color="k", lw=0.5)
plt.tight_layout(); plt.show()
print("per-level correlation between the two panels (over units x features): " + ", ".join(f"L{l}: {np.corrcoef(D2[:, feature_level == l].ravel(), pred[:, feature_level == l].ravel())[0, 1]:.3f}" for l in range(5)))
print(f"gradient on the ideal rows: fraction of |G1 row| along the row's own direction (norm change) vs off it, by level: " + ", ".join(f"L{l}: {np.mean([abs(G1[r] @ W1[r]) / np.linalg.norm(W1[r]) / np.linalg.norm(G1[r]) for r in range(62) if feature_level[r] == l]):.2f}" for l in range(5)))


**Findings (section 1).** The gradient at a random W2 with a +-1 readout has a large label-independent part (logit rms 0.12-0.19, so the
residual is not -y/2), which pushes all of a unit's input weights the same way and hides the structure; the statistics below are for the
label part of the W2 descent (gradient at f -> 0 with the same gates), the actual gradient being the sum of the two.
- **Leaf columns: every unit wants to read the leaves with the sign of its readout.** The descent on W2[u, (m, pole)] has the sign of
  Wout[u] x pole on 96-98% of the entries (63-93% for the actual gradient). A positive-readout unit grows +relu(a_m) and shrinks
  relu(-a_m), i.e. it reads a_m itself: the label-aligned leaf reading of the lens theory, present from the first step.
- **Internal columns follow the unit's leaf half-readings.** The descent on the feature of side `pole` of node j correlates with
  Wout[u] x S_u(j, pole) at 0.70-0.80 for the root, 0.61-0.65 at level 1, 0.45-0.52 at level 2, 0.32-0.39 at level 3 (over units x
  features, five seeds; the actual gradient: 0.02-0.2). So the converse of the design holds in the lens sense: with ideal features the
  gradient on W2 is set by what the unit already reads from the leaves under each node, side by side, exactly the quantity the designed
  cue blocks were built from. It holds less well with depth, where fewer leaves enter each half-reading and other terms compete.
- **But the first step does not build vetoes.** A veto needs the two poles of a node pushed in opposite directions, with the negative one on
  the side where the unit's leaf reading is opposed to its readout. The two poles get opposite-sign descents on 37-53% of the (unit,
  node) pairs at the root and 28-33% at level 3, around or below chance (0.50), and the predicted sides on 18-35% / 14-17%. A random W2 reads
  both sides of a node with the same sign as often as not, so the first step adds side-dependent weights, not the one-sided vetoes of the
  converged runs; those need the readout and the gates to co-evolve (the lineage of runs 22-25: the W2 sign pattern picks the family).
- **Magnitudes.** The descent rms on internal features falls by a factor 2.1-2.2 per level (1.3e-4 / 6.3e-5 / 2.9e-5 / 1.4e-5), but the
  ideal rows were scaled by 2^(-level), and the W2 gradient is proportional to the feature's norm; per unit norm the ladder is flat
  (1.3 / 1.26 / 1.16 / 1.12 x 1e-4). The gradient on the ideal rows themselves is 85-92% off their own direction (rotation, not norm change).


## 1b. Ideal first layer, W2 = 0, alternating readout, ReLU'(0) = 1

With W2 = 0 every unit's preactivation is exactly 0, so with the usual subgradient ReLU'(0) = 0 nothing reaches W2 (and W1 and W_out get
no gradient either: dL/dW1 is proportional to W2, and h2 = 0 kills dL/dW_out). With ReLU'(0) = 1 the units are "open" everywhere and the
descent on W2 is `-dL/dW2[u, r] = -Wout[u] E[resid f_r] = Wout[u] E[y f_r] / 2` exactly (f = 0, so resid = -y/2): the plain label
correlation of each feature. Analytically that is `Wout[u] x pole x E|a_m| / 32` on a leaf feature and 0 on every internal feature
(`E[y relu(+-a_j)] = 0` for an internal node: the label is balanced on both sides). The cell checks this, and then takes one small descent
step on W2 (the gates now depend on the data) and looks at the gradient again, since the question is whether the internal structure
can appear at all from this start.


In [ ]:
W1 = ideal_first_layer(); W2 = np.zeros((60, 62)); Wo = np.array([[(-1.0) ** u for u in range(60)]])
loss0, (G1, G2, Go) = population_gradient(W1, W2, Wo, relu_sub1=False); print(f"W2 = 0 with ReLU'(0) = 0: loss {loss0:.4f}, |dL/dW1| {np.linalg.norm(G1):.1e}, |dL/dW2| {np.linalg.norm(G2):.1e}, |dL/dWout| {np.linalg.norm(Go):.1e}")
loss0, (G1, G2, Go) = population_gradient(W1, W2, Wo, relu_sub1=True); D2 = -G2; print(f"W2 = 0 with ReLU'(0) = 1: loss {loss0:.4f}, |dL/dW1| {np.linalg.norm(G1):.1e}, |dL/dW2| {np.linalg.norm(G2):.1e}, |dL/dWout| {np.linalg.norm(Go):.1e}")
leaf_cols = [r for r in range(62) if feature_level[r] == 4]; int_cols = [r for r in range(62) if feature_level[r] < 4]
pred_leaf = np.array([[Wo[0, u] * ROW_ID[r][1] * np.linalg.norm(W1[r]) for r in leaf_cols] for u in range(60)]) * np.mean(np.abs(A[:, 15:])) / 64      # Wout x pole x ||row|| x E|a_m| / 64  (E[y f_r] / 2 with E[y relu(pole a_m)] = pole E|a_m| / 32)
print(f"leaf columns: sign = Wout x pole on {(np.sign(D2[:, leaf_cols]) == np.sign(pred_leaf)).mean():.3f}; descent / analytic prediction: mean ratio {np.mean(D2[:, leaf_cols] / pred_leaf):.3f} (sd {np.std(D2[:, leaf_cols] / pred_leaf):.3f})")
print(f"internal columns: rms {np.sqrt(np.mean(D2[:, int_cols] ** 2)):.2e} vs leaf columns rms {np.sqrt(np.mean(D2[:, leaf_cols] ** 2)):.2e}; per unit the internal entries are identical across units up to the readout sign (corr between units 0 and 2: {np.corrcoef(D2[0, int_cols], D2[2, int_cols])[0, 1]:.3f}), so they are the sample's label-feature correlations, not unit structure")
print(f"rank of the descent matrix (singular values / largest): {np.round(np.linalg.svd(D2, compute_uv=False)[:4] / np.linalg.svd(D2, compute_uv=False)[0], 4)}")
# one small descent step on W2 from zero, then the gradient again (gates now data-dependent; W1 and Wout kept). For the internal columns the
# question is whether the two poles of a node are pushed differently (a side contrast, the seed of a veto) or the same way (a scale change).
def pole_split(D):
    """rms over (unit, internal node) of the difference and of the sum of the descent on the node's two pole features."""
    dif, tot = [], []
    for j in range(15):
        rp = [r for r in range(62) if ROW_ID[r] == (j, 1)][0]; rm = [r for r in range(62) if ROW_ID[r] == (j, -1)][0]; dif.append(D[:, rp] - D[:, rm]); tot.append(D[:, rp] + D[:, rm])
    return np.sqrt(np.mean(np.array(dif) ** 2)), np.sqrt(np.mean(np.array(tot) ** 2))
for max_w2 in (0.005, 0.05):
    W2s = D2 / np.abs(D2).max() * max_w2
    loss1, (G1b, G2b, Gob) = population_gradient(W1, W2s, Wo, relu_sub1=True); D2b = -G2b
    _, (_, G2l, _) = population_gradient(W1, W2s, 1e-3 * Wo, relu_sub1=True); D2l = -G2l * 1e3                                   # label part (f -> 0, same gates)
    c = cue_readings(W1, W2s); S = half_readings(c); act = float((np.maximum(np.maximum(x[:100_000] @ W1.T, 0) @ W2s.T, 0) > 0).mean())
    f = forward_logit(W1, W2s, Wo); d_act, t_act = pole_split(D2b[:, :]); d_lab, t_lab = pole_split(D2l)
    print(f"\nafter a step with max |W2| = {max_w2}: loss {loss1:.5f}, logit rms {np.sqrt(np.mean(f ** 2)):.3f}, corr(f, v . x) {np.corrcoef(f, x[:100_000] @ v)[0, 1]:.3f}, units active on {act:.2f} of points; |dL/dW1| {np.linalg.norm(G1b):.1e}")
    print(f"   units' leaf readings: side contrast C_u(j) rms {np.sqrt(np.mean((S[:, :, 0] - S[:, :, 1]) ** 2)):.1e} vs total reading rms {np.sqrt(np.mean((S[:, :, 0] + S[:, :, 1]) ** 2)):.1e} (balanced: every unit reads all 16 leaves alike)")
    print(f"   descent on internal columns, actual: pole difference rms {d_act:.1e} vs pole sum rms {t_act:.1e}; label part: difference {d_lab:.1e} vs sum {t_lab:.1e}; leaf columns rms {np.sqrt(np.mean(D2b[:, leaf_cols] ** 2)):.1e}")
    print(f"   sign of the internal descent vs the unit's readout: actual {np.mean(np.sign(D2b[:, int_cols]) == np.sign(Wo[0])[:, None]):.2f} agree, label part {np.mean(np.sign(D2l[:, int_cols]) == np.sign(Wo[0])[:, None]):.2f} agree")


**Findings (section 1b).**
- **With the standard subgradient nothing moves**: W2 = 0 gives zero gradient on all three matrices (the units sit exactly at the kink, and
  W1 / W_out need a path through W2 or a nonzero h2). With ReLU'(0) = 1 only W2 gets a gradient, and it is exactly the analytic one: on the
  leaf columns `Wout[u] x pole x ||row|| x E|a_m| / 64` (sign right on 100% of entries, ratio to the formula 1.000 +- 0.045), on the internal
  columns zero (what is measured there, rms 3e-5 against 1.1e-4 on the leaves, is the sample's label-feature correlation, identical for every
  unit up to the readout sign). The descent matrix has rank one: every unit takes the same step, `Wout[u] x (the leaf-sign reader)`.
- **After that step the gates are informative but symmetric, and no internal structure follows.** One step gives every unit a reading of all
  16 leaves with the same weight (side contrast 1% of the total reading, i.e. noise), the logit becomes a v-reader (f correlates 0.44 with
  v . x, units active on exactly half the points), and the next descent on the internal columns is again the same on both poles of every
  node (pole difference 5-6x below the pole sum, at the noise level; the label part's sign agrees with the readout on 50% of entries, i.e. it
  carries nothing). The pole-sum part is the logit-shrinking term (its sign is opposite to the readout on 97-100% of entries, and at the
  larger step the loss has overshot).
- **Reading.** Starting from W2 = 0 the second layer is a symmetric fixed subspace of the flow: all units stay identical up to their
  readout sign, so the network is effectively a two-unit reader of the coherent leaf signal v, and the side contrasts that push a probe
  toward an internal node, or a unit toward a veto, can never appear. The converse of the design holds only once the units read the leaves
  unequally, which is what a random W2 (section 1) supplies and the designed blocks supply by construction. For the theorem this is the
  precise sense in which the random init is needed: not for the leaf reading, which the gradient produces on its own, but for the
  symmetry breaking between units that turns leaf readings into side contrasts.


## 2. Run 60's W2 and W_out with the first layer collapsed onto v: where do the probes get pushed?

Every probe row is s x pole x alpha v (v = u_15 + ... + u_30, pole = the probe's designed pole, s = +-1 a global sign convention) and every cue
row is pole x (alpha v + epsilon u_m) (its designed identity, as the cue init pole x (v + u_j) of runs 37-62). The gradient on the 30 probe
rows is projected on the 31 normals. The lens prediction: the probe of node j is pushed along u_j by its two units' side-contrasts (the
designed blocks, read through the epsilon components), by an amount proportional to epsilon and with the designed pole, and along v by the
coherent leaf signal, independent of epsilon.

**The scalar cannot be chosen by minimising the log loss: the loss is flat.** At every scalar multiple of v the design's quadruple symmetry
makes the four units of a node cancel almost exactly (the logit rms stays at 0.02 for alpha from 0.02 to 1, E[f y] is of order 1e-5 with a
sign set by s), so the log loss is log 2 + O(1e-5) for every alpha and the minimiser is alpha = 0, where the probes are zero rows with no
gradient at all. Two degeneracies feed this. At epsilon = 0 every unit's cue block is side-balanced, so its reading of the cue features
(all +-alpha v) cancels, its own probe contributes -alpha relu(+-v . x) and the unit is dead on every point. And if the two probes of a
node were given the same sign, the two units (B, +) and (B, -) of the node would have identical inputs and opposite readouts and the logit
would vanish identically. The cells below show the flat loss curves and then analyse the gradient at the init scale, alpha = sqrt 2 / |v|
(row norm sqrt 2, as run 60's rows), with s = -1 (the convention with E[f y] > 0); the direction of the probe push is what matters, and
it is checked against alpha and s at the end.


In [ ]:
def v_first_layer(alpha, eps, probe_sign=-1):
    W1 = np.zeros((62, 100))
    for p_ in PROBE_ROWS: W1[p_] = probe_sign * HIERARCHY_TARGETS[p_][1] * alpha * v                                 # probes: s x pole x alpha v (the two probes of a node opposite)
    for (m, pole), r in CUE_ID.items(): W1[r] = pole * (alpha * v + eps * U[m])
    return W1
EPS_LIST = (0.0, 0.02, 0.05, 0.1, 0.2, 0.5); ALPHA0 = float(np.sqrt(2) / np.linalg.norm(v)); grid = np.concatenate([np.linspace(0.0, 0.3, 16), np.linspace(0.4, 2.0, 9)])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
for k, s_ in enumerate((-1, +1)):
    for eps in (0.0, 0.1, 0.5):
        losses = [population_gradient(v_first_layer(a, eps, s_), W2_60, Wo_60)[0] for a in grid]; ax[k].plot(grid, np.array(losses) - np.log(2), ".-", ms=3, lw=0.8, label=f"eps {eps}: min at alpha {grid[int(np.argmin(losses))]:.2f}")
    ax[k].set_xlabel("alpha"); ax[k].set_ylabel("log loss - log 2"); ax[k].set_title(f"probe sign s = {s_:+d}: rows = s pole alpha v (probes), pole (alpha v + eps u_m) (cues)", fontsize=9); ax[k].legend(fontsize=7); ax[k].axvline(ALPHA0, color="k", lw=0.5)
plt.tight_layout(); plt.show()
print(f"alpha used below: {ALPHA0:.4f} (row norm sqrt 2); the loss-minimising alpha is 0 for every eps and sign")


In [ ]:
def probe_push_table(eps, alpha=ALPHA0, probe_sign=-1):
    W1 = v_first_layer(alpha, eps, probe_sign); loss, (G1, G2, Go) = population_gradient(W1, W2_60, Wo_60); D = -G1[PROBE_ROWS]; Cp = D @ U.T                       # push of each probe on the 31 normals
    vhat = v / np.linalg.norm(v); rows = []
    for p in PROBE_ROWS:
        j, pole = HIERARCHY_TARGETS[p]; leaf = np.sqrt(np.mean(Cp[p, 15:] ** 2)); internal = Cp[p, :15]; others = np.delete(internal, j)
        rows.append({"probe": p, "level": NODE_LEVEL[j], "target": f"{'+' if pole > 0 else '-'}u{j}", "push on target (signed by pole)": pole * Cp[p, j], "push along v": D[p] @ vhat, "leaf rms": leaf, "largest other internal |push|": np.abs(others).max(), "node of largest other": int(np.delete(np.arange(15), j)[np.abs(others).argmax()]),
                     "off-span push": np.linalg.norm(D[p] - Cp[p] @ U), "|push|": np.linalg.norm(D[p])})
    t = pd.DataFrame(rows); t["target / leaf rms"] = t["push on target (signed by pole)"] / t["leaf rms"]; t["target / |push|"] = t["push on target (signed by pole)"] / t["|push|"]
    return loss, t, Cp
def forward_hidden(W1, n=100_000):
    with torch.no_grad(): return torch.relu(torch.relu(X[:n] @ torch.as_tensor(W1, dtype=torch.float32, device=dev).T) @ torch.as_tensor(W2_60, dtype=torch.float32, device=dev).T).cpu().numpy()
summary = []
for eps, alpha, s_ in [(e, ALPHA0, -1) for e in EPS_LIST] + [(0.1, ALPHA0, +1), (0.1, 0.1, -1), (0.1, 1.0, -1)]:
    loss, t, Cp = probe_push_table(eps, alpha, s_); g = t.groupby("level"); act = float((forward_hidden(v_first_layer(alpha, eps, s_)) > 0).mean())
    summary.append({"eps": eps, "alpha": round(alpha, 3), "s": s_, "loss": round(loss, 5), "unit active frac": round(act, 3), "probes whose largest internal push is the target, with the designed pole": int(((t["push on target (signed by pole)"] > 0) & (t["push on target (signed by pole)"].abs() > t["largest other internal |push|"])).sum()),
                    "target push / leaf rms by level 0/1/2/3 (mean)": " / ".join(f"{g['target / leaf rms'].mean()[l]:.2f}" for l in range(4)), "target push / |push| by level": " / ".join(f"{g['target / |push|'].mean()[l]:.2f}" for l in range(4)),
                    "push along v / |push| (mean)": round(float((t["push along v"] / t["|push|"]).mean()), 3), "off-span / |push| (mean)": round(float((t["off-span push"] / t["|push|"]).mean()), 3)})
print(pd.DataFrame(summary).to_string(index=False))
loss, t, Cp = probe_push_table(0.1); print(f"\neps = 0.1, per probe:"); print(t.round(5).to_string(index=False))


In [ ]:
# The probes' pushes on the 31 normals for eps = 0.1 (rows: probes in design order, columns: nodes), signed so that + means toward the designed pole of the probe's own target.
loss, t, Cp = probe_push_table(0.1); M = Cp / np.abs(Cp).max(1, keepdims=True)
fig, ax = plt.subplots(figsize=(14, 7)); im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"); plt.colorbar(im, ax=ax, label="push on u_n / max over n (per probe)")
ax.set_xticks(range(31)); ax.set_xticklabels([f"u{n} (L{NODE_LEVEL[n]})" for n in range(31)], rotation=90, fontsize=7); ax.set_yticks(range(30)); ax.set_yticklabels([f"p{p} -> {'+' if HIERARCHY_TARGETS[p][1] > 0 else '-'}u{HIERARCHY_TARGETS[p][0]}" for p in range(30)], fontsize=7)
for kk in (1, 3, 7, 15): ax.axvline(kk - 0.5, color="k", lw=0.5)
ax.set_title("eps = 0.1: push of every probe row on every ODT normal (rows = probes, columns = nodes; the leaf columns carry the coherent v pull)"); plt.tight_layout(); plt.show()


**Findings (section 2).** At the init scale (row norm sqrt 2) the population push on every probe row points at its designed node with its
designed pole, and at nothing else.
- **Every probe is pushed to its target.** For epsilon >= 0.05 all 30 probes have their largest internal component on their designed node
  with the designed pole (22 of 30 at epsilon 0.02); at epsilon 0.1 the target carries 87 / 77 / 63 / 53% of the push's norm by level, the
  rest being sampling noise (next point). The push along v is zero to three decimals: the coherent leaf signal cancels on the probes within
  each node's quadruple (two units of opposite readout read the same probe through blocks of opposite sign), so unlike a random row, a
  designed probe feels no pull toward v or toward any leaf at first order.
- **The other components are noise.** Doubling the sample from 400k to 1.6M leaves the target push at 6.5e-5 and halves the leaf rms
  (7.5e-6 -> 4.1e-6), the off-span part (6.6e-5 -> 3.3e-5) and the other-internal rms (7.8e-6 -> 4.1e-6), all as 1 / sqrt n; another seed
  gives the same. So the "target / leaf rms" ratios in the table (18 / 13 / 10 / 6 by level at epsilon 0.1, 24 / 16 / 11 / 8 at 0.2) are
  lower bounds set by the sample, and the population push is along the target alone. The ladder by level is the design's contrast count
  (2^(4 - level) cues under the node), and the push grows with epsilon up to 0.2 and saturates by 0.5 (the epsilon term starts to
  dominate the cue features). It also grows as alpha shrinks (24 at alpha 0.1, 18 at 0.35, 9 at 1.0, in leaf-rms units): the smaller the
  v component, the larger the overlap of the probe's gate with its units' gates.
- **Sign convention and scale.** The probe sign s only flips the sign of E[f y] (both conventions give the same push structure, the loss
  is flat either way); the loss-minimising alpha is 0 for every epsilon, so "the scalar that minimises the log loss" is not a usable
  definition at this configuration. The design's job is visible instead: the cue components of size epsilon are the only thing the
  probes see, and they see them as a push straight at their node.


## 3. Ideal first layer with equal norms, W2 = a x random + b x (run 60's probe links), run 60's readout

W1 = the 62 features +-u_j, every row of norm sqrt 2. W2[u, r] = b x (-1) on the designed probe link of unit u (run 60's: unit u reads its
own probe with -1; the probe's pole sets the unit's side: a veto on +u_j leaves the side a_j < 0) and a x N(0, 2/62) everywhere else
(the other probe columns and all 32 cue columns). W_out = run 60's path-sign readout +-0.816. At a = 0 every unit is dead (its only input
is -relu) and nothing moves; at b = 0 it is section 1 with run 60's readout. For each (a, b) the cells report, on the **label part** of the
gradient (f -> 0, same gates) unless stated:
- the designed probe links: is the descent negative (the veto strengthens)?
- the other probe columns: do the ancestor features of the unit's node get a veto on the wrong side (descent < 0 on the feature of the
  side of the ancestor the unit's region is *not* on, >= 0 on the other), as in the converged runs?
- the cue columns: do they follow the unit's leaf half-readings (corr with Wout x S_u(j, pole)), and for the level-3 units does the cue of
  the unit's child leaf get the converged cue veto (descent < 0 on the cue of the sign opposite to the readout)?
- the rows: how much of each probe row's descent is along itself (norm growth), how much is the side-signed ancestor tilt of the converged
  nets (positive = toward the manufactured bias), how much goes to the leaves.


In [ ]:
P_LINK = np.where(W2_60[:, :30] != 0, -1.0, 0.0); P_LINK = np.concatenate([P_LINK, np.zeros((60, 32))], axis=1)                        # run 60's probe links (-1), nothing else
unit_probe = np.array([int(np.nonzero(P_LINK[u, :30])[0][0]) for u in range(60)]); unit_node_ = np.array([HIERARCHY_TARGETS[p][0] for p in unit_probe]); unit_side = np.array([-HIERARCHY_TARGETS[p][1] for p in unit_probe])     # a veto on +u_j leaves the side a_j < 0
def W1_equal(norm=np.sqrt(2.0)):
    W1 = np.zeros((62, 100))
    for r, (n, p_) in ROW_ID.items(): W1[r] = p_ * U[n] * norm
    return W1
def feat(n, pole): return [r for r in range(62) if ROW_ID[r] == (n, pole)][0]
def analyse_grid(a, b, seed=0, verbose=False):
    rng = np.random.default_rng(seed); W1 = W1_equal(); W2 = a * rng.standard_normal((60, 62)) * np.sqrt(2 / 62); W2[P_LINK != 0] = b * P_LINK[P_LINK != 0]; Wo = Wo_60
    loss, (G1, G2, Go) = population_gradient(W1, W2, Wo); _, (G1l, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo); D2 = -G2l * 1e3; D1 = -G1l * 1e3; D1a = -G1
    f = forward_logit(W1, W2, Wo); act = float((np.maximum(np.maximum(x[:100_000] @ W1.T, 0) @ W2.T, 0) > 0).mean())
    # designed links
    link_desc = np.array([D2[u, unit_probe[u]] for u in range(60)])
    # ancestor vetoes: for unit u (node j, side s) and ancestor a of j: wrong side of a = -side_a(j)
    wrong_neg, right_nonneg, both = [], [], []
    for u in range(60):
        j = unit_node_[u]
        for a_ in node_ancestors(j):
            sa = side_of(a_, j); dw, dr = D2[u, feat(a_, -sa)], D2[u, feat(a_, sa)]; wrong_neg.append(dw < 0); right_nonneg.append(dr >= 0); both.append(dw < 0 and dr > dw)
    # cue columns vs half-readings; level-3 cue veto
    c = cue_readings(W1, W2); S = half_readings(c); int_cols = [r for r in range(62) if feature_level[r] < 4]; cue_cols = [r for r in range(62) if feature_level[r] == 4]
    pred_int = np.array([[Wo[0, u] * S[u, ROW_ID[r][0], 0 if ROW_ID[r][1] > 0 else 1] for r in int_cols] for u in range(60)]); corr_int = np.corrcoef(D2[:, int_cols].ravel(), pred_int.ravel())[0, 1]
    pred_cue = np.array([[Wo[0, u] * ROW_ID[r][1] for r in cue_cols] for u in range(60)]); cue_sign = float((np.sign(D2[:, cue_cols]) == np.sign(pred_cue)).mean())
    cv = []
    for u in range(60):
        j = unit_node_[u]
        if NODE_LEVEL[j] == 3:
            m = 2 * j + (2 if unit_side[u] > 0 else 1); sgn = int(np.sign(Wo[0, u])); cv.append(D2[u, feat(m, -sgn)] < 0 and D2[u, feat(m, sgn)] > D2[u, feat(m, -sgn)])
    # rows: probes
    rows = []
    for p_ in PROBE_ROWS:
        j, pole = HIERARCHY_TARGETS[p_]; d = D1[p_]; w = W1[p_] / np.linalg.norm(W1[p_]); along = d @ w; rest = d - along * w; Cp = rest @ U.T
        anc = node_ancestors(j); tilt = sum(Cp[a_] * side_of(a_, j) for a_ in anc) if anc else np.nan
        rows.append({"along": along, "|rest|": np.linalg.norm(rest), "ancestor tilt (side-signed)": tilt, "|anc|": np.linalg.norm(Cp[anc]) if anc else np.nan, "|leaves|": np.linalg.norm(Cp[15:]), "|other internal|": np.linalg.norm(np.delete(Cp[:15], [j] + anc))})
    t = pd.DataFrame(rows)
    rec = {"a": a, "b": b, "loss": round(loss, 4), "logit rms": round(float(np.sqrt(np.mean(f ** 2))), 3), "active": round(act, 2), "|dL/dW1| / |dL/dW2| / |dL/dWo| (actual)": f"{np.linalg.norm(G1):.1e} / {np.linalg.norm(G2):.1e} / {np.linalg.norm(Go):.1e}",
           "probe links: descent < 0": round(float((link_desc < 0).mean()), 2), "mean link descent / rms(W2 descent)": round(float(link_desc.mean() / np.sqrt(np.mean(D2 ** 2))), 2) if np.abs(D2).max() > 0 else np.nan,
           "ancestor vetoes: wrong side < 0 / right side >= 0 / both": f"{np.mean(wrong_neg):.2f} / {np.mean(right_nonneg):.2f} / {np.mean(both):.2f}", "cue cols: sign = Wout x pole": round(cue_sign, 2), "internal cols: corr with Wout x S": round(float(corr_int), 2),
           "level-3 cue veto formed": round(float(np.mean(cv)), 2), "probe rows: along / rest (mean)": f"{t['along'].mean():.1e} / {t['|rest|'].mean():.1e}", "probe rows: rest split anc / leaves / other / off-span": f"{np.nanmean(t['|anc|']):.1e} / {t['|leaves|'].mean():.1e} / {t['|other internal|'].mean():.1e} / {np.mean([np.linalg.norm(D1[p_] - (D1[p_] @ U.T) @ U) for p_ in PROBE_ROWS]):.1e}",
           "ancestor tilt side-signed > 0 (frac of probes with ancestors)": round(float((t["ancestor tilt (side-signed)"].dropna() > 0).mean()), 2)}
    return rec
recs = [analyse_grid(a, b) for a in (0.0, 0.1, 0.3, 1.0) for b in (0.0, 0.3, 1.0, 3.0) if not (a == 0 and b == 0)]
pd.set_option("display.max_colwidth", 60); print(pd.DataFrame(recs).to_string(index=False))


In [ ]:
# the W2 descent (label part) in the feature basis for a = 0.3, b = 1 (seed 0), units in design order (4 per node, nodes in heap order), features sorted by node; and the probe-link column marked
rng = np.random.default_rng(0); W1 = W1_equal(); W2 = 0.3 * rng.standard_normal((60, 62)) * np.sqrt(2 / 62); W2[P_LINK != 0] = 1.0 * P_LINK[P_LINK != 0]
_, (_, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo_60); D2 = -G2l * 1e3; order = np.lexsort((-np.array([ROW_ID[r][1] for r in range(62)]), np.array([ROW_ID[r][0] for r in range(62)])))
fig, ax = plt.subplots(figsize=(16, 8)); M = D2[:, order] / np.abs(D2).max(); im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"); plt.colorbar(im, ax=ax, label="descent on W2 / max")
ax.set_xticks(range(62)); ax.set_xticklabels(feature_label[order], rotation=90, fontsize=6); ax.set_yticks(range(60)); ax.set_yticklabels([f"U{u} n{unode[u]} {'+' if Wo_60[0, u] > 0 else '-'}" for u in range(60)], fontsize=5)
for kk in np.cumsum([(feature_level[order] == l).sum() for l in range(4)]): ax.axvline(kk - 0.5, color="k", lw=0.5)
for u in range(60): ax.plot(list(order).index(unit_probe[u]), u, "k.", ms=2)
ax.set_title("a = 0.3, b = 1: descent on W2 (label part); dots = the unit's designed probe link"); plt.tight_layout(); plt.show()


**Findings (section 3).** Statistics on the label part of the gradient; the random scale a sets the overall size of every gradient
(linearly: all three norms scale with a) and the link scale b sets the units' activity (0.46 of points at b = 0, 0.23-0.27 at b = 3) and
little else. a = 0 is dead, as predicted.
- **The designed probe links get nothing.** The descent on the -1 link is negative on 45-55% of the units (chance) and its mean is 0 to 7% of
  the W2 descent rms, falling to 0 as b grows. A unit's only strong reader of its probe vetoes it, so the unit is dead where the probe fires
  and the link's gradient, which needs both active, vanishes: the self-gating freeze seen in run 60 after epoch 8. The first step neither
  strengthens nor weakens the veto; the link's later growth in the trained runs comes from the other units that recruit the probe as an
  ancestor veto, not from its own unit.
- **No ancestor vetoes either.** For a unit of node j and an ancestor a, the feature on the side of a that the unit's region is not on gets a
  negative descent on 47-50% of the pairs and the other side a non-negative one on 40-43%: chance. The one-sided ancestor vetoes of the
  converged nets (54 / 36 / 30 on the root, level 1, level 2 in run 53) are not a first-order consequence of ideal features plus the
  designed links; in the runs they appear between epochs 20 and 80, once the regions are set and the readouts hand over.
- **What the first step does do is read the leaves and follow the half-readings.** The cue columns get the sign Wout x pole on 90-97% of
  entries (the label-aligned leaf reading, as in sections 1 and 1b), and the internal columns correlate with Wout x S_u(j, pole) at 0.55-0.62.
  The level-3 "cue veto" of the converged leaf detectors (negative on the cue of the sign opposite to the readout, for the unit's child
  leaf) is formed in 100% of the units, but it is the same leaf-reading rule applied to that cue, not an extra structure.
- **The probe rows do not grow; they are pulled toward the leaves.** The descent along a probe row is 1e-6 against 2e-4 off it (the same
  freeze: the row's own feature is gated out of its unit's gradient), and the off-row part is 90% on the leaves, with the ancestors at a
  seventh of that and the side-signed ancestor tilt of the converged nets only weakly favoured (positive on 61-68% of the probes).
- **Reading.** With the first layer already ideal, the converse of the design holds for the leaf reading and the half-reading structure of
  W2, which the gradient produces immediately, but not for the vetoes, neither the designed probe link nor the ancestor vetoes, which the
  first step leaves at chance. The veto architecture is a product of the dynamics, the readout growth and the hand-over between levels,
  and not of the gradient at an ideal point.


## 4. Section 3 with a +-c v component on the probes

Same W2 (a = 0.3 random part, b = 1 designed links, seed 0) and readout as section 3, W1 as there (every row +-u_j at norm sqrt 2) except
that probe row p becomes `pole u_j sqrt 2 + s_p c v`, v = u_15 + ... + u_30 (|v| = 4, so the added part has norm 4c: 0.4 / 1.2 / 4 for
c = 0.1 / 0.3 / 1 against the 1.41 on the node; rows are not renormalised). Sign patterns s_p: all +, all -, alternating (+ for even
probes, - for odd: the two probes of a node opposite, i.e. s_p = pole), and random +-1 (seed 4). A probe with a v component fires where
s_p v . x is large, so its gate, and through the -1 link its unit's gate, now correlate with the coherent leaf signal and hence with the
label. Reported, on the label part of the gradient: the W2 statistics of section 3, and for the probe rows the descent decomposed into the
component on the own node (signed by the pole: + keeps the probe on it), on v (signed by s_p: + grows the row's own v part, - removes it),
along the row, on the leaves and on the ancestors.


In [ ]:
def analyse_config(W1, W2, Wo, vsign=None):
    loss, (G1, G2, Go) = population_gradient(W1, W2, Wo); _, (G1l, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo); D2 = -G2l * 1e3; D1 = -G1l * 1e3
    f = forward_logit(W1, W2, Wo); act = float((np.maximum(np.maximum(x[:100_000] @ W1.T, 0) @ W2.T, 0) > 0).mean()); row_act = float((x[:100_000] @ W1[PROBE_ROWS].T > 0).mean())
    link_desc = np.array([D2[u, unit_probe[u]] for u in range(60)]); wrong_neg, both = [], []
    for u in range(60):
        j = unit_node_[u]
        for a_ in node_ancestors(j):
            sa = side_of(a_, j); dw, dr = D2[u, feat(a_, -sa)], D2[u, feat(a_, sa)]; wrong_neg.append(dw < 0); both.append(dw < 0 and dr > dw)
    c = cue_readings(W1, W2); S = half_readings(c); int_cols = [r for r in range(62) if feature_level[r] < 4]; cue_cols = [r for r in range(62) if feature_level[r] == 4]
    pred_int = np.array([[Wo[0, u] * S[u, ROW_ID[r][0], 0 if ROW_ID[r][1] > 0 else 1] for r in int_cols] for u in range(60)]); corr_int = np.corrcoef(D2[:, int_cols].ravel(), pred_int.ravel())[0, 1]
    pred_cue = np.array([[Wo[0, u] * ROW_ID[r][1] for r in cue_cols] for u in range(60)]); cue_sign = float((np.sign(D2[:, cue_cols]) == np.sign(pred_cue)).mean())
    vhat = v / np.linalg.norm(v); rows = []
    for p_ in PROBE_ROWS:
        j, pole = HIERARCHY_TARGETS[p_]; d = D1[p_]; w = W1[p_] / np.linalg.norm(W1[p_]); Cp = d @ U.T; anc = node_ancestors(j); sp = 1 if vsign is None else vsign[p_]
        rows.append({"own node (signed)": pole * Cp[j], "v (signed by s_p)": sp * (d @ vhat), "along row": d @ w, "|d|": np.linalg.norm(d), "|leaves|": np.linalg.norm(Cp[15:]), "|anc|": np.linalg.norm(Cp[anc]) if anc else np.nan, "anc tilt (side-signed)": sum(Cp[a_] * side_of(a_, j) for a_ in anc) if anc else np.nan,
                     "|other internal|": np.linalg.norm(np.delete(Cp[:15], [j] + anc)), "off-span": np.linalg.norm(d - Cp @ U)})
    t = pd.DataFrame(rows)
    return {"loss": round(loss, 4), "logit rms": round(float(np.sqrt(np.mean(f ** 2))), 3), "units active": round(act, 2), "probe rows active": round(row_act, 2), "|dL/dW1| / |dL/dW2|": f"{np.linalg.norm(G1):.1e} / {np.linalg.norm(G2):.1e}",
            "links: descent < 0 / mean rel.": f"{(link_desc < 0).mean():.2f} / {link_desc.mean() / np.sqrt(np.mean(D2 ** 2)):+.2f}", "anc vetoes: wrong < 0 / both": f"{np.mean(wrong_neg):.2f} / {np.mean(both):.2f}", "cue sign = Wout x pole": round(cue_sign, 2), "internal corr Wout x S": round(float(corr_int), 2),
            "probe descent: own node / v / along row / |d| (mean)": f"{t['own node (signed)'].mean():+.1e} / {t['v (signed by s_p)'].mean():+.1e} / {t['along row'].mean():+.1e} / {t['|d|'].mean():.1e}", "probes with own-node descent > 0": round(float((t["own node (signed)"] > 0).mean()), 2), "probes with v descent < 0 (v part removed)": round(float((t["v (signed by s_p)"] < 0).mean()), 2),
            "|leaves| / |anc| / |other| / off-span (mean)": f"{t['|leaves|'].mean():.1e} / {np.nanmean(t['|anc|']):.1e} / {t['|other internal|'].mean():.1e} / {t['off-span'].mean():.1e}", "anc tilt side-signed > 0": round(float((t["anc tilt (side-signed)"].dropna() > 0).mean()), 2)}
rng = np.random.default_rng(0); W2_ref = 0.3 * rng.standard_normal((60, 62)) * np.sqrt(2 / 62); W2_ref[P_LINK != 0] = 1.0 * P_LINK[P_LINK != 0]
PATTERNS = {"all +": np.ones(30), "all -": -np.ones(30), "alternating (= pole)": np.array([1 if p_ % 2 == 0 else -1 for p_ in range(30)]), "random": np.random.default_rng(4).choice([-1, 1], 30)}
recs = []
for c_ in (0.0, 0.1, 0.3, 1.0):
    for name, sgn in (PATTERNS.items() if c_ > 0 else [("(none)", np.ones(30))]):
        W1 = W1_equal()
        for p_ in PROBE_ROWS: W1[p_] = W1[p_] + sgn[p_] * c_ * v
        recs.append({"c": c_, "pattern": name, "|probe row|": round(float(np.linalg.norm(W1[0])), 2), **analyse_config(W1, W2_ref, Wo_60, vsign=sgn)})
pd.set_option("display.max_colwidth", 70); print(pd.DataFrame(recs).to_string(index=False))


**Findings (section 4).** The v component changes what the gates see, not what the first step builds in W2; and it decides, weakly,
whether the probes stay on their nodes.
- **W2: same structure as section 3, with the half-reading signal drowned.** The designed links stay at chance (descent negative on
  45-63% of units, mean within 1% of the rms) and so do the ancestor vetoes (wrong side negative on 49-57%). The cue columns keep the sign
  Wout x pole but less reliably as c grows (0.93 at c = 0 -> 0.80-0.84 at 0.1 -> 0.6 at 0.3 -> 0.5 at 1), and the correlation of the internal
  columns with Wout x S_u(j, pole) collapses (0.58 -> 0.14-0.29 -> 0.04-0.14 -> 0.01-0.10): with +-c v on its probe, a unit's gate is set by
  v . x through the -1 link rather than by its random leaf readings, so the side contrasts that the half-readings predict no longer govern
  the internal columns. The pole-signed (alternating) pattern is the extreme case: it turns the designed links into a v-reader (logit rms
  0.12 / 0.16 / 0.32 and |dL/dW2| 10x section 3's at c = 0.1 / 0.3 / 1) with the wrong sign (loss 0.698 / 0.701 / 0.713 above log 2),
  "all +" into one with the right sign (loss 0.695 / 0.694 / 0.6925, the only configuration below log 2).
- **Probe rows: still frozen along themselves, still pulled to the leaves.** The descent along the row is 1e-6 or below in every case and
  the leaf part is 4-6e-4 of the 5-7e-4 total (at c = 1 with a same-sign pattern the whole descent shrinks to 2e-4). The own-node component is
  1e-5 to 3e-5, a few percent of the leaf pull, with a pattern-dependent sign: positive (the probe is held on its node) for 63-67% of the
  probes with the pole-signed pattern and for 50-53% with "all -", negative for most probes with "all +" at c = 0.1-0.3 (37-47% positive).
  The v component itself is removed by the descent under the pole-signed and random patterns (v part signed by s_p: -1.3e-4 / -0.4e-4 /
  -0.3e-4 for alternating, -0.8e-4 / -1.2e-4 / -1.2e-4 for random at c = 0.1 / 0.3 / 1) and grown under "all +" (+0.4e-4 at c <= 0.3).
- **Reading.** A v component on an ideal probe does not help the gradient find the designed second layer; it mostly re-gates the units on
  the coherent leaf signal. Whether it is self-correcting depends on the sign pattern: with the two probes of a node opposite in v
  (the pattern of the cue init pole x (v + u_j)) the step removes the v part and keeps the probes on their nodes, with a same-sign pattern
  the probes drift. All effects on the rows are small next to the leaf pull, which remains the dominant force on an ideal probe at the
  first step, as in section 3.


## 5. One unit: weight c on one probe, +1 on every other row

W1 as in section 3 (every row +-u_j at norm sqrt 2), a single second-layer unit with W2[r] = +1 on all rows except the connected probe,
which gets c in {+0.1, +3, -0.1, -3}, readout +1 (the readout sign only flips every descent). The connected probe is +u_j or -u_j for
j = 0 (root), 1 and 7 (level 3). The gradient on W2 is a 62-vector in the feature basis; the cells show it per configuration (actual and
label part), with the unit's activity and the logit, and tabulate the descent on the connected probe, on its pole partner, on the
ancestors of j by side, on the other internal features and on the leaves.


In [ ]:
W1 = W1_equal(); Wo1 = np.array([[1.0]])
def one_unit(c_, probe):
    W2 = np.ones((1, 62)); W2[0, probe] = c_; return W2
cfgs = [(c_, feat(j, pole)) for j in (0, 1, 7) for pole in (1, -1) for c_ in (0.1, 3.0, -0.1, -3.0)]
recs = []; D_act, D_lab = [], []
for c_, probe in cfgs:
    W2 = one_unit(c_, probe); loss, (G1, G2, Go) = population_gradient(W1, W2, Wo1); _, (_, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo1); d, dl = -G2[0], -G2l[0] * 1e3; D_act.append(d); D_lab.append(dl)
    f = forward_logit(W1, W2, Wo1); act = float((np.maximum(x[:100_000] @ W1.T, 0) @ W2.T > 0).mean()); j, pole = ROW_ID[probe]; partner = feat(j, -pole); anc = node_ancestors(j)
    rec = {"probe": feature_label[probe], "c": c_, "loss": round(loss, 3), "logit mean / rms": f"{f.mean():.2f} / {np.sqrt(np.mean(f ** 2)):.2f}", "unit active": round(act, 3),
           "descent on the probe: actual / label": f"{d[probe]:+.1e} / {dl[probe]:+.1e}", "on its partner": f"{d[partner]:+.1e} / {dl[partner]:+.1e}",
           "ancestors, side of j / other side (label part, mean)": (f"{np.mean([dl[feat(a_, side_of(a_, j))] for a_ in anc]):+.1e} / {np.mean([dl[feat(a_, -side_of(a_, j))] for a_ in anc]):+.1e}" if anc else "-"),
           "other internal (label part rms)": f"{np.sqrt(np.mean([dl[r] ** 2 for r in range(62) if feature_level[r] < 4 and ROW_ID[r][0] not in [j] + anc])):.1e}", "leaves: sign = pole (label part) / rms": f"{np.mean([np.sign(dl[r]) == ROW_ID[r][1] for r in range(62) if feature_level[r] == 4]):.2f} / {np.sqrt(np.mean([dl[r] ** 2 for r in range(62) if feature_level[r] == 4])):.1e}",
           "actual: all entries < 0": bool((d < 0).all()), "actual rms": f"{np.sqrt(np.mean(d ** 2)):.1e}"}
    recs.append(rec)
pd.set_option("display.max_colwidth", 60); print(pd.DataFrame(recs).to_string(index=False))
order = np.lexsort((-np.array([ROW_ID[r][1] for r in range(62)]), np.array([ROW_ID[r][0] for r in range(62)])))
fig, ax = plt.subplots(2, 1, figsize=(16, 9))
for k, (D, title) in enumerate([(np.array(D_act), "actual descent -dL/dW2"), (np.array(D_lab), "label part (f -> 0, same gate)")]):
    M = D[:, order] / np.abs(D).max(1, keepdims=True); im = ax[k].imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"); plt.colorbar(im, ax=ax[k], label="descent / max |descent| per row")
    ax[k].set_xticks(range(62)); ax[k].set_xticklabels(feature_label[order], rotation=90, fontsize=6); ax[k].set_yticks(range(len(cfgs))); ax[k].set_yticklabels([f"{feature_label[pr]} c={c_:+.1f}" for c_, pr in cfgs], fontsize=6); ax[k].set_title(title, fontsize=9)
    for kk in np.cumsum([(feature_level[order] == l).sum() for l in range(4)]): ax[k].axvline(kk - 0.5, color="k", lw=0.5)
    for i, (c_, pr) in enumerate(cfgs): ax[k].plot(list(order).index(pr), i, "k.", ms=3)
plt.tight_layout(); plt.show()


In [ ]:
# Variant: the same unit with the other 61 weights scaled to w_other = 1/61 (their sum of features is then ~0.06), so that c = +-3 on the probe actually gates the unit.
recs2 = []
for w_other in (1.0, 1 / 61):
    for c_, probe in [(c_, feat(j, pole)) for j in (0, 7) for pole in (1, -1) for c_ in (0.1, 3.0, -0.1, -3.0)]:
        W2 = np.full((1, 62), w_other); W2[0, probe] = c_; loss, (G1, G2, Go) = population_gradient(W1, W2, Wo1); _, (_, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo1); d, dl = -G2[0], -G2l[0] * 1e3
        f = forward_logit(W1, W2, Wo1); h = np.maximum(x[:100_000] @ W1.T, 0) @ W2.T; act = float((h > 0).mean()); j, pole = ROW_ID[probe]; partner = feat(j, -pole); anc = node_ancestors(j)
        gate_corr = float(np.corrcoef((h[:, 0] > 0).astype(float), np.sign(A[:100_000, j]) * pole)[0, 1]) if 0 < act < 1 else np.nan
        recs2.append({"w_other": round(w_other, 4), "probe": feature_label[probe], "c": c_, "loss": round(loss, 4), "logit mean": round(float(f.mean()), 3), "unit active": round(act, 3), "corr(gate, pole sign(a_j))": round(gate_corr, 3) if not np.isnan(gate_corr) else "-",
                      "label-part descent: probe / partner": f"{dl[probe]:+.1e} / {dl[partner]:+.1e}", "ancestors side of j / other side (mean)": (f"{np.mean([dl[feat(a_, side_of(a_, j))] for a_ in anc]):+.1e} / {np.mean([dl[feat(a_, -side_of(a_, j))] for a_ in anc]):+.1e}" if anc else "-"),
                      "leaves under j, pole side / other side (sign = pole frac)": f"{np.mean([np.sign(dl[feat(m, q)]) == q for m in LEAVES_UNDER[j] if side_of(j, m) == pole for q in (1, -1)]):.2f} / {np.mean([np.sign(dl[feat(m, q)]) == q for m in LEAVES_UNDER[j] if side_of(j, m) == -pole for q in (1, -1)]):.2f}" if j < 15 else "-",
                      "leaves rms (label)": f"{np.sqrt(np.mean([dl[r] ** 2 for r in range(62) if feature_level[r] == 4])):.1e}", "internal rms excl. j (label)": f"{np.sqrt(np.mean([dl[r] ** 2 for r in range(62) if feature_level[r] < 4 and ROW_ID[r][0] != j])):.1e}", "actual: entries < 0": f"{(d < 0).mean():.2f}"})
print(pd.DataFrame(recs2).to_string(index=False))


**Findings (section 5).** With +1 on the other 61 rows the unit is active on every point (its preactivation is a sum of 61 non-negative
features, about 3.5 on average, that no single c in [-3, 3] can cancel), so **the W2 gradient does not depend on c or on the probe at all**.
- **Actual descent: a uniform shrink.** The logit is +3.3 to +3.6 everywhere (loss 1.68-1.84, rising with c), so every one of the 62 entries
  gets a negative descent of about 2.6e-2, the same in all 24 configurations; c only shifts the logit and the loss.
- **Label part: the leaf reading and nothing else.** The 32 leaf columns get the sign of their pole on 100% of entries (rms 1.8e-3); the 30
  internal columns, the connected probe and its partner included, sit at the sample's noise level (3-9e-5, identical across c because the
  sample is fixed). An always-open unit is a linear reader, and a linear reader's gradient is the feature-label correlation, zero on every
  internal node.
- **When the probe does gate the unit (variant: other weights 1/61, c = -3)** the unit is dead where the probe fires (active on 55% of
  points, gate correlation -0.90 with the probe's side), and then: the descent on the probe's own weight is exactly 0 (-1e-6: the freeze of
  sections 3-4, from both sides at once), the leaf reading persists at a reduced size (1.0-1.3e-3, the vetoed side contributing less), and
  the ancestors of the node still get no side asymmetry (+5.5e-5 / +6.3e-5 for +u7). c = +3 and c = +-0.1 leave the unit always open and
  reproduce the first case with a smaller logit.
- **Reading.** For a single unit the sign and size of its probe weight matter only through the gate; once the unit is gated by the probe,
  that weight stops moving and the rest of the gradient is the leaf reading restricted to the open side. No internal-node structure arises
  from one unit and ideal features, whatever c is.


## 6. One unit: weight c on one probe, 1/100 on the 32 cue rows, 0 on the other probes

Same W1 (every row +-u_j at norm sqrt 2) and readout +1. The unit's preactivation is `c sqrt 2 relu(pole a_j) + 0.01 sqrt 2 sum_m |a_m|`
(both cue rows of a leaf carry +0.01, so the cue part reads |a_m|, about 0.015 on average, with no label information): for c > 0 the unit
is open everywhere, for c < 0 it is closed where the probe fires strongly enough, i.e. on the probe's side of u_j once |c| is large. The
cells report the W2 descent (label part; the logit is small here so the actual descent is nearly the same) on the probe, on the cue
columns split by the side of u_j the leaf lies on, on the ancestors of j by side and on the other internal features, plus the descent on
the probe row itself.


In [ ]:
recs6 = []; D6 = []; cfg6 = [(c_, feat(j, pole)) for j in (0, 1, 7) for pole in (1, -1) for c_ in (0.1, 3.0, -0.1, -3.0)]
for c_, probe in cfg6:
    W2 = np.zeros((1, 62)); W2[0, CUE_ROWS] = 0.01; W2[0, probe] = c_
    loss, (G1, G2, Go) = population_gradient(W1, W2, Wo1); _, (_, G2l, _) = population_gradient(W1, W2, 1e-3 * Wo1); d, dl = -G2[0], -G2l[0] * 1e3; D6.append(dl)
    f = forward_logit(W1, W2, Wo1); h = np.maximum(x[:100_000] @ W1.T, 0) @ W2.T; act = float((h > 0).mean()); j, pole = ROW_ID[probe]; partner = feat(j, -pole); anc = node_ancestors(j)
    gate_corr = float(np.corrcoef((h[:, 0] > 0).astype(float), np.sign(A[:100_000, j]) * pole)[0, 1]) if 0 < act < 1 else np.nan
    same = [feat(m, q) for m in LEAVES_UNDER[j] if side_of(j, m) == pole for q in (1, -1)] if j < 15 else []; other = [feat(m, q) for m in LEAVES_UNDER[j] if side_of(j, m) == -pole for q in (1, -1)]
    outside = [r for r in CUE_ROWS if ROW_ID[r][0] not in LEAVES_UNDER[j]]
    cue_side = lambda cols: f"{np.mean([dl[r] * ROW_ID[r][1] for r in cols]):+.1e} ({np.mean([np.sign(dl[r]) == ROW_ID[r][1] for r in cols]):.2f})" if cols else "-"
    dprobe = -G1[probe]; Cp = dprobe @ U.T
    recs6.append({"probe": feature_label[probe], "c": c_, "loss": round(loss, 4), "logit mean / rms": f"{f.mean():.3f} / {np.sqrt(np.mean(f ** 2)):.3f}", "unit active": round(act, 3), "corr(gate, probe side)": round(gate_corr, 2) if not np.isnan(gate_corr) else "-",
                  "descent on the probe / partner": f"{dl[probe]:+.1e} / {dl[partner]:+.1e}", "cue cols pole-signed mean (sign = pole frac): leaves on the probe's side": cue_side(same), "leaves on the other side of j": cue_side(other), "leaves outside j's subtree": cue_side(outside),
                  "ancestors: side of j / other side": (f"{np.mean([dl[feat(a_, side_of(a_, j))] for a_ in anc]):+.1e} / {np.mean([dl[feat(a_, -side_of(a_, j))] for a_ in anc]):+.1e}" if anc else "-"), "other internal rms": f"{np.sqrt(np.mean([dl[r] ** 2 for r in range(62) if feature_level[r] < 4 and ROW_ID[r][0] not in [j] + anc])):.1e}",
                  "probe row descent: along itself / on leaves / |total|": f"{dprobe @ (W1[probe] / np.linalg.norm(W1[probe])):+.1e} / {np.linalg.norm(Cp[15:]):.1e} / {np.linalg.norm(dprobe):.1e}"})
pd.set_option("display.max_colwidth", 60); print(pd.DataFrame(recs6).to_string(index=False))
order = np.lexsort((-np.array([ROW_ID[r][1] for r in range(62)]), np.array([ROW_ID[r][0] for r in range(62)]))); D6 = np.array(D6)
fig, ax = plt.subplots(figsize=(16, 5)); M = D6[:, order] / np.abs(D6).max(1, keepdims=True); im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"); plt.colorbar(im, ax=ax, label="label-part descent / max per row")
ax.set_xticks(range(62)); ax.set_xticklabels(feature_label[order], rotation=90, fontsize=6); ax.set_yticks(range(len(cfg6))); ax.set_yticklabels([f"{feature_label[pr]} c={c_:+.1f}" for c_, pr in cfg6], fontsize=6)
for kk in np.cumsum([(feature_level[order] == l).sum() for l in range(4)]): ax.axvline(kk - 0.5, color="k", lw=0.5)
for i, (c_, pr) in enumerate(cfg6): ax.plot(list(order).index(pr), i, "k.", ms=3)
ax.set_title("one unit, c on the probe (dot), 0.01 on the cues: descent on W2 (label part)"); plt.tight_layout(); plt.show()


**Findings (section 6).** Here c matters, through the gate, and what it produces is the designed cue block.
- **c > 0: an open unit, a full leaf reading.** The unit is active everywhere; every cue column gets the leaf reading with the sign of its
  pole (pole-signed mean +1.8e-3, 100% of entries, on both sides of u_j and outside j's subtree alike); the probe column and the internal
  columns are at the sample's noise level (5-9e-5); the logit is 0.02 at c = 0.1 and 0.19 at c = 3 (loss 0.705), where the actual descent
  also shrinks the probe row along itself (-1.6e-2) and pulls it to the leaves (2e-2).
- **c = -3: a unit vetoed by its probe, and a one-sided cue reading.** The unit is closed on the probe's side of u_j (active on 52% of
  points, gate correlation -0.97 with the probe's side). The descent on the probe's own weight is exactly 0 (the freeze again), and the
  cue columns split by side: the leaves on the open side of j keep the full reading (+1.8e-3, 100%), the leaves on the probe's side get
  nothing (3e-5 to 9e-5 pole-signed, sign agreement 50-88%, i.e. noise), and the leaves outside j's subtree get half (+9.1e-4: the unit is
  open on half of their cells). **This is the design's cue block B_j**: a side-signed contrast over the leaves under j, produced by one
  step of the gradient from a probe gate and a uniform cue reading. The lens formula then says this block pushes the probe toward u_j,
  which closes the loop the design was built to seed: probe gate -> side contrast in the cues -> push on the probe.
- **c = -0.1: the same, weakly.** Active on 89% of points, gate correlation -0.35; the probe weight's descent is 0 within noise, the cue
  reading on the probe's side is reduced to 1.4e-3 against 1.8e-3 on the other side.
- **Ancestors and other internal columns** stay at noise (5-10e-5) in every case; the asymmetry between the sides of an ancestor is at most
  1-2 noise units (+4.8e-5 / -0.9e-5 for the root as seen by a -3-gated u_1 unit), not a veto.
- **Reading.** A single unit with ideal features needs two things to build a side contrast: a gate on one node (a negative probe weight
  large enough to close the unit on that side) and a reading of the leaves (the small positive cue weights). Given both, one step writes
  B_j into its cue weights, restricted to the leaves under j and signed by side, and freezes the probe weight. The design put B_j in by
  hand; the gradient would have produced it from a veto and a uniform leaf reading. What the gradient still does not produce at one step
  is the next level's vetoes.
